# Setup


In [ ]:
import json
import re
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from scipy import stats

from statsmodels.nonparametric.smoothers_lowess import lowess as sm_lowess

# Constants


In [ ]:
DATA_DIR = Path("/Users/casperkristiansson/Thesis Data/logs_4")

DIAGNOSTICS_DIR = Path("figures/diagnostics")
QQ_DIR = DIAGNOSTICS_DIR / "qq"
DRIFT_DIR = DIAGNOSTICS_DIR / "drift"
SUMMARY_DIR = DIAGNOSTICS_DIR

ARTIFACTS_DIR = Path("figures/artifacts")
TABLE_DIR = ARTIFACTS_DIR / "tables"
FIGURE_DIR = ARTIFACTS_DIR / "figures"

for path in (QQ_DIR, DRIFT_DIR, SUMMARY_DIR, TABLE_DIR, FIGURE_DIR):
    path.mkdir(parents=True, exist_ok=True)

FORMATS = ["hdf5", "zarr", "tiledb", "root"]
CODECS = ["gzip", "lz4", "zstd"]
PATTERNS = ["slice", "full"]
ROWS = [(codec, pattern) for codec in CODECS for pattern in PATTERNS]

RUN_RX = re.compile(r"^(?P<fmt>hdf5|zarr|tiledb|root)_(?P<codec>gzip|lz4|zstd)_(?P<pattern>slice|full)_(?P<runid>\d{8}-\d{6})\.json$")
CW_RX = re.compile(r"^(?P<fmt>hdf5|zarr|tiledb|root)_(?P<codec>gzip|lz4|zstd)_(?P<pattern>slice|full)_cloudwatch\.json$")


# Helper Functions


In [ ]:
def parse_run_meta(name: str) -> dict[str, str] | None:
    match = RUN_RX.match(name)
    return match.groupdict() if match else None


def parse_cloudwatch_meta(name: str) -> dict[str, str] | None:
    match = CW_RX.match(name)
    return match.groupdict() if match else None


def load_runs(path: Path) -> pd.DataFrame:
    with open(path, "r") as handle:
        payload = json.load(handle)
    frame = pd.DataFrame(payload["data"])
    if "i" not in frame.columns:
        frame["i"] = np.arange(1, len(frame) + 1)
    frame["time_sec"] = frame["t_total_ns"] / 1e9
    frame["ln_time"] = np.log(frame["time_sec"])
    return frame


def qq_envelope(n: int, draws: int = 2000, seed: int = 979969114) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
    rng = np.random.default_rng(seed)
    samples = np.sort(rng.standard_normal(size=(draws, n)), axis=1)
    lower = np.percentile(samples, 2.5, axis=0)
    upper = np.percentile(samples, 97.5, axis=0)
    theoretical = stats.norm.ppf((np.arange(1, n + 1) - 0.5) / n)
    return theoretical, lower, upper


def lag1_autocorr(values: np.ndarray) -> float:
    centered = values - values.mean()
    denom = np.dot(centered, centered)
    if denom == 0:
        return 0.0
    return float(np.dot(centered[:-1], centered[1:]) / denom)


def smooth_series(x: np.ndarray, y: np.ndarray) -> np.ndarray:
    return sm_lowess(y, x, frac=0.3, return_sorted=False)

def nice_ylim(ymin: float, ymax: float, pad_frac: float = 0.1, floor: float | None = None) -> tuple[float, float]:
    if not np.isfinite(ymin) or not np.isfinite(ymax):
        base = floor if floor is not None else 0.0
        return base, base + 1.0
    if ymax <= ymin:
        ymax = ymin + 1.0
    span = ymax - ymin
    pad = pad_frac * span if span > 0 else 1.0
    low = ymin - pad
    high = ymax + pad
    if floor is not None:
        low = max(floor, low)
    return low, high


def garwood_rate_ci(k: float, exposure: float, alpha: float = 0.05) -> tuple[float, float]:
    if not np.isfinite(exposure) or exposure <= 0 or not np.isfinite(k) or k < 0:
        return np.nan, np.nan
    lower = 0.0 if k == 0 else 0.5 * stats.chi2.ppf(alpha / 2, 2 * k)
    upper = 0.5 * stats.chi2.ppf(1 - alpha / 2, 2 * (k + 1))
    return lower / exposure, upper / exposure


# Diagnostic Summaries


In [ ]:
def plot_qq_ln_times(frame: pd.DataFrame, meta: dict[str, str], outdir: Path) -> None:
    z = (frame["ln_time"] - frame["ln_time"].mean()) / frame["ln_time"].std(ddof=1)
    z_sorted = np.sort(z.values)
    n = len(z_sorted)
    theoretical, envelope_lo, envelope_hi = qq_envelope(n)
    statistic, p_value = stats.shapiro(frame["ln_time"].values)
    plt.figure(figsize=(5.5, 5.5))
    plt.fill_between(theoretical, envelope_lo, envelope_hi, alpha=0.2, label="95% envelope")
    plt.plot(theoretical, z_sorted, marker="o", linestyle="", markersize=3, label="sample")
    limit = np.nanmax(np.abs(np.concatenate([theoretical, z_sorted])))
    limit = float(np.ceil(limit * 1.05 * 10) / 10)
    plt.plot([-limit, limit], [-limit, limit], linestyle="--", linewidth=1)
    title = f"Q–Q ln(time): {meta['fmt']}/{meta['codec']}/{meta['pattern']} | n={n} | Shapiro p={p_value:.3g}"
    plt.title(title)
    plt.xlabel("Theoretical z (N(0,1))")
    plt.ylabel("Sample z (standardized ln time)")
    plt.tight_layout()
    filename = f"qq_{meta['fmt']}_{meta['codec']}_{meta['pattern']}.png"
    plt.savefig(outdir / filename, dpi=150)
    plt.close()


def plot_drift_ln_times(frame: pd.DataFrame, meta: dict[str, str], outdir: Path) -> None:
    x = frame["i"].values
    y = frame["ln_time"].values
    geometric = float(np.exp(y.mean()))
    rho1 = lag1_autocorr(y)
    plt.figure(figsize=(7, 4))
    plt.plot(x, y, marker="o", linestyle="-", linewidth=1)
    y_smooth = smooth_series(x, y)
    plt.plot(x, y_smooth, linewidth=2)
    plt.axhline(np.log(geometric), linestyle="--", linewidth=1)
    title = f"Run-order ln(time): {meta['fmt']}/{meta['codec']}/{meta['pattern']} | n={len(x)} | ρ₁={rho1:.2f}"
    plt.title(title)
    plt.xlabel("Repetition index")
    plt.ylabel("ln(time [s])")
    plt.tight_layout()
    filename = f"drift_{meta['fmt']}_{meta['codec']}_{meta['pattern']}.png"
    plt.savefig(outdir / filename, dpi=150)
    plt.close()


def build_run_summary(data_dir: Path, qq_dir: Path, drift_dir: Path) -> pd.DataFrame:
    fields = [
        "file",
        "format",
        "codec",
        "pattern",
        "run_id",
        "n",
        "geometric_mean_sec",
        "p95_sec",
        "shapiro_W",
        "shapiro_p",
        "rho1",
        "spearman_rho",
        "spearman_p",
    ]
    summaries: list[dict[str, object]] = []
    for path in sorted(data_dir.iterdir()):
        if not path.is_file() or not path.name.endswith(".json") or "cloudwatch" in path.name:
            continue
        meta = parse_run_meta(path.name)
        if not meta:
            continue
        frame = load_runs(path)
        plot_qq_ln_times(frame, meta, qq_dir)
        plot_drift_ln_times(frame, meta, drift_dir)
        gm = float(np.exp(frame["ln_time"].mean()))
        p95 = float(np.exp(np.quantile(frame["ln_time"], 0.95)))
        shapiro_stat, shapiro_p = stats.shapiro(frame["ln_time"].values)
        rho1 = lag1_autocorr(frame["ln_time"].values)
        rho_s, p_s = stats.spearmanr(frame["i"].values, frame["ln_time"].values)
        summaries.append({
            "file": path.name,
            "format": meta["fmt"],
            "codec": meta["codec"],
            "pattern": meta["pattern"],
            "run_id": meta["runid"],
            "n": int(len(frame)),
            "geometric_mean_sec": gm,
            "p95_sec": p95,
            "shapiro_W": float(shapiro_stat),
            "shapiro_p": float(shapiro_p),
            "rho1": float(rho1),
            "spearman_rho": float(rho_s) if np.isfinite(rho_s) else np.nan,
            "spearman_p": float(p_s) if np.isfinite(p_s) else np.nan,
        })
    if not summaries:
        return pd.DataFrame(columns=fields)
    summary = pd.DataFrame(summaries, columns=fields)
    summary = summary.sort_values(["format", "codec", "pattern", "run_id"]).reset_index(drop=True)
    summary["p95_over_gm"] = summary["p95_sec"] / summary["geometric_mean_sec"]
    summary["non_normal_flag"] = summary["shapiro_p"] < 0.05
    return summary


def format_run_summary_table(summary: pd.DataFrame) -> pd.DataFrame:
    if summary.empty:
        columns = [
            "Format",
            "Codec",
            "Pattern",
            "n",
            r"Shapiro $p$",
            r"Non-normal (p<0.05)",
            r"$\rho_1$",
            r"$\mathrm{p95}/\mathrm{GM}$",
            r"Spearman $p$",
        ]
        return pd.DataFrame(columns=columns)
    table = summary.loc[:, [
        "format",
        "codec",
        "pattern",
        "n",
        "shapiro_p",
        "non_normal_flag",
        "rho1",
        "p95_over_gm",
        "spearman_p",
    ]].copy()
    table = table.rename(columns={
        "format": "Format",
        "codec": "Codec",
        "pattern": "Pattern",
        "n": "n",
        "shapiro_p": r"Shapiro $p$",
        "non_normal_flag": r"Non-normal (p<0.05)",
        "rho1": r"$\rho_1$",
        "p95_over_gm": r"$\mathrm{p95}/\mathrm{GM}$",
        "spearman_p": r"Spearman $p$",
    })
    table[r"Shapiro $p$"] = table[r"Shapiro $p$"].map(lambda x: f"{x:.3g}")
    table[r"Non-normal (p<0.05)"] = table[r"Non-normal (p<0.05)"].map(lambda v: r"\textbf{Yes}" if v else "No")
    table[r"$\rho_1$"] = table[r"$\rho_1$"].map(lambda x: f"{x:.2f}")
    table[r"$\mathrm{p95}/\mathrm{GM}$"] = table[r"$\mathrm{p95}/\mathrm{GM}$"].map(lambda x: f"{x:.2f}")
    table[r"Spearman $p$"] = table[r"Spearman $p$"].map(lambda x: "nan" if pd.isna(x) else f"{x:.3g}")
    table = table.sort_values(["Codec", "Pattern", "Format"]).reset_index(drop=True)
    return table


In [ ]:
run_summary = build_run_summary(DATA_DIR, QQ_DIR, DRIFT_DIR)
run_summary.to_csv(SUMMARY_DIR / "diagnostics_summary_full.csv", index=False)
run_summary_table = format_run_summary_table(run_summary)
run_summary_table.to_csv(SUMMARY_DIR / "diagnostics_summary_table.csv", index=False)

# Appendix Grids


In [ ]:
def latest_runs(data_dir: Path) -> dict[tuple[str, str, str], tuple[str, Path]]:
    latest: dict[tuple[str, str, str], tuple[str, Path]] = {}
    for path in data_dir.iterdir():
        if not path.is_file() or not path.name.endswith(".json") or "cloudwatch" in path.name:
            continue
        meta = parse_run_meta(path.name)
        if not meta:
            continue
        key = (meta["fmt"], meta["codec"], meta["pattern"])
        run_id = meta["runid"]
        record = latest.get(key)
        if record is None or run_id > record[0]:
            latest[key] = (run_id, path)
    return latest


def generate_appendix_grids(data_dir: Path, dest_dir: Path) -> None:
    latest = latest_runs(data_dir)
    cells_data: dict[tuple[str, str, str], dict[str, object]] = {}
    for fmt in FORMATS:
        for codec, pattern in ROWS:
            key = (fmt, codec, pattern)
            if key not in latest:
                continue
            run_id, path = latest[key]
            frame = load_runs(path)
            cells_data[key] = {
                "frame": frame,
                "run_id": run_id,
                "shapiro_p": stats.shapiro(frame["ln_time"].values)[1],
                "rho1": lag1_autocorr(frame["ln_time"].values),
            }
    if not cells_data:
        return
    fig, axes = plt.subplots(len(ROWS), len(FORMATS), figsize=(14, 18), sharex=True, sharey=True)
    for r, (codec, pattern) in enumerate(ROWS):
        for c, fmt in enumerate(FORMATS):
            ax = axes[r, c]
            key = (fmt, codec, pattern)
            if key not in cells_data:
                ax.axis("off")
                continue
            frame = cells_data[key]["frame"]
            z = (frame["ln_time"] - frame["ln_time"].mean()) / frame["ln_time"].std(ddof=1)
            z_sorted = np.sort(z.values)
            theoretical, envelope_lo, envelope_hi = qq_envelope(len(z_sorted))
            ax.fill_between(theoretical, envelope_lo, envelope_hi, alpha=0.15)
            ax.plot(theoretical, z_sorted, marker="o", linestyle="", markersize=2)
            lim = 3.0
            ax.plot([-lim, lim], [-lim, lim], linestyle="--", linewidth=0.8)
            ax.set_xlim(-lim, lim)
            ax.set_ylim(-lim, lim)
            if r == 0:
                ax.set_title(fmt, fontsize=10)
            if c == 0:
                ax.set_ylabel(f"{codec}/{pattern}", fontsize=9)
            ax.text(0.98, 0.02, f"p={cells_data[key]['shapiro_p']:.3g}", transform=ax.transAxes, ha="right", va="bottom", fontsize=8)
    fig.suptitle("Q–Q plots of ln(time) by codec/pattern and format", fontsize=12)
    fig.text(0.5, 0.005, "Theoretical z (N(0,1))", ha="center")
    fig.text(0.005, 0.5, "Sample z (standardized ln time)", va="center", rotation="vertical")
    fig.tight_layout(rect=(0.03, 0.03, 1, 0.96))
    plt.savefig(dest_dir / "appendix_QQ_grid_6x4.png", dpi=300)
    plt.savefig(dest_dir / "appendix_QQ_grid_6x4.pdf")
    plt.close(fig)
    fig, axes = plt.subplots(len(ROWS), len(FORMATS), figsize=(14, 18), sharex=True, sharey=False)
    for r, (codec, pattern) in enumerate(ROWS):
        for c, fmt in enumerate(FORMATS):
            ax = axes[r, c]
            key = (fmt, codec, pattern)
            if key not in cells_data:
                ax.axis("off")
                continue
            frame = cells_data[key]["frame"]
            x = frame["i"].values
            y = frame["ln_time"].values
            quantiles = np.quantile(y, [0.02, 0.98])
            if not np.isfinite(quantiles).all() or quantiles[1] <= quantiles[0]:
                ymin, ymax = float(y.min()), float(y.max())
            else:
                ymin, ymax = float(quantiles[0]), float(quantiles[1])
            span = max(ymax - ymin, 1e-3)
            pad = 0.05 * span
            ax.set_ylim(ymin - pad, ymax + pad)
            ax.plot(x, y, marker="o", linestyle="-", linewidth=0.8, markersize=2)
            y_smooth = smooth_series(x, y)
            ax.plot(x, y_smooth, linewidth=1.5)
            geometric = float(np.exp(y.mean()))
            ax.axhline(np.log(geometric), linestyle="--", linewidth=0.8)
            if r == 0:
                ax.set_title(fmt, fontsize=10)
            if c == 0:
                ax.set_ylabel(f"{codec}/{pattern}", fontsize=9)
            if r == len(ROWS) - 1:
                ax.set_xlabel("rep", fontsize=9)
            ax.text(0.98, 0.02, f"ρ₁={cells_data[key]['rho1']:.2f}", transform=ax.transAxes, ha="right", va="bottom", fontsize=8)
    fig.suptitle("Run-order traces of ln(time) by codec/pattern and format", fontsize=12)
    fig.text(0.005, 0.5, "ln(time [s])", va="center", rotation="vertical")
    fig.tight_layout(rect=(0.03, 0.03, 1, 0.96))
    plt.savefig(dest_dir / "appendix_Drift_grid_6x4.png", dpi=300)
    plt.savefig(dest_dir / "appendix_Drift_grid_6x4.pdf")
    plt.close(fig)


In [ ]:
generate_appendix_grids(DATA_DIR, DIAGNOSTICS_DIR)


# Request Intensity


In [ ]:
def latest_run_by_mtime(data_dir: Path) -> dict[tuple[str, str, str], Path]:
    latest: dict[tuple[str, str, str], tuple[float, Path]] = {}
    for path in data_dir.iterdir():
        if not path.is_file() or path.suffix != ".json" or "cloudwatch" in path.name:
            continue
        meta = parse_run_meta(path.name)
        if not meta:
            continue
        key = (meta["fmt"], meta["codec"], meta["pattern"])
        record = latest.get(key)
        mtime = path.stat().st_mtime
        if record is None or mtime > record[0]:
            latest[key] = (mtime, path)
    return {key: value for key, (_, value) in latest.items()}


def latest_cloudwatch_by_mtime(data_dir: Path) -> dict[tuple[str, str, str], Path]:
    latest: dict[tuple[str, str, str], tuple[float, Path]] = {}
    for path in data_dir.iterdir():
        if not path.is_file() or path.suffix != ".json" or "_cloudwatch" not in path.name:
            continue
        meta = parse_cloudwatch_meta(path.name)
        if not meta:
            continue
        key = (meta["fmt"], meta["codec"], meta["pattern"])
        record = latest.get(key)
        mtime = path.stat().st_mtime
        if record is None or mtime > record[0]:
            latest[key] = (mtime, path)
    return {key: value for key, (_, value) in latest.items()}


def compute_request_intensity_table(data_dir: Path) -> pd.DataFrame:
    run_files = latest_run_by_mtime(data_dir)
    cloudwatch_files = latest_cloudwatch_by_mtime(data_dir)
    rows: list[dict[str, object]] = []
    for key in sorted(set(run_files) & set(cloudwatch_files)):
        fmt, codec, pattern = key
        with open(run_files[key], "r") as handle:
            run_payload = json.load(handle)
        n_included = int(len(run_payload["data"]))
        with open(cloudwatch_files[key], "r") as handle:
            cw_payload = json.load(handle)
        gets = float(cw_payload.get("GetRequestsSum", np.nan))
        bytes_downloaded = float(cw_payload.get("BytesDownloadedSum", np.nan))
        gib = bytes_downloaded / (2 ** 30) if np.isfinite(bytes_downloaded) else np.nan
        gets_per_rep = gets / n_included if n_included > 0 else np.nan
        gets_per_gib = gets / gib if np.isfinite(gib) and gib > 0 else np.nan
        rep_lo, rep_hi = garwood_rate_ci(gets, n_included)
        gib_lo, gib_hi = garwood_rate_ci(gets, gib)
        rows.append({
            "format": fmt,
            "codec": codec,
            "pattern": pattern,
            "GetRequestsSum": gets,
            "BytesDownloadedSum_GiB": gib,
            "n_included": n_included,
            "GETs_per_rep": gets_per_rep,
            "GETs_per_rep_lo": rep_lo,
            "GETs_per_rep_hi": rep_hi,
            "GETs_per_GiB": gets_per_gib,
            "GETs_per_GiB_lo": gib_lo,
            "GETs_per_GiB_hi": gib_hi,
            "run_file": run_files[key].name,
            "cw_file": cloudwatch_files[key].name,
        })
    frame = pd.DataFrame(rows)
    if frame.empty:
        return frame
    frame = frame.sort_values(["codec", "pattern", "format"]).reset_index(drop=True)
    return frame


def plot_request_intensity_series(frame: pd.DataFrame, metric: str, lower: str, upper: str, prefix: str, ylabel: str) -> None:
    for codec in CODECS:
        for pattern in PATTERNS:
            subset = frame[(frame["codec"] == codec) & (frame["pattern"] == pattern)].copy()
            if subset.empty:
                continue
            subset = subset.set_index("format").reindex(FORMATS).dropna(subset=[metric]).reset_index()
            if subset.empty:
                continue
            x = np.arange(len(subset))
            values = subset[metric].to_numpy(float)
            lower_bounds = np.maximum(0.0, subset[lower].to_numpy(float))
            upper_bounds = np.maximum(0.0, subset[upper].to_numpy(float))
            errors = [np.maximum(0.0, values - lower_bounds), np.maximum(0.0, upper_bounds - values)]
            ymin = float(np.nanmin(np.minimum(lower_bounds, values)))
            ymax = float(np.nanmax(np.maximum(upper_bounds, values)))
            lo, hi = nice_ylim(ymin, ymax, pad_frac=0.08, floor=0.0)
            plt.figure(figsize=(6.0, 3.2))
            ax = plt.gca()
            ax.errorbar(x, values, yerr=errors, fmt="o", capsize=4, linewidth=1)
            ax.set_xticks(x)
            ax.set_xticklabels(subset["format"])
            ax.set_ylim(lo, hi)
            ax.set_ylabel(ylabel)
            ax.set_title(f"Request intensity — {codec}/{pattern}")
            ax.yaxis.grid(True, alpha=0.3)
            for xi, yi in zip(x, values):
                ax.text(xi, yi, f"{yi:.3g}", ha="center", va="bottom", fontsize=8)
            plt.tight_layout()
            plt.savefig(FIGURE_DIR / f"{prefix}_{codec}_{pattern}.png", dpi=180)
            plt.close()


def plot_request_intensity_grid(frame: pd.DataFrame, metric: str, lower: str, upper: str, ylabel: str, prefix: str, minimum_ci_fraction: float = 0.003) -> None:
    fig, axes = plt.subplots(nrows=len(CODECS), ncols=len(PATTERNS), figsize=(9.5, 9.0), sharex=False, sharey=False)
    for r, codec in enumerate(CODECS):
        for c, pattern in enumerate(PATTERNS):
            ax = axes[r, c] if len(CODECS) > 1 else axes[c]
            subset = frame[(frame["codec"] == codec) & (frame["pattern"] == pattern)].set_index("format").reindex(FORMATS).reset_index()
            subset = subset.dropna(subset=[metric])
            ax.set_title(f"{codec} / {pattern}", fontsize=10)
            if subset.empty:
                ax.axis("off")
                continue
            x = np.arange(len(subset))
            values = subset[metric].to_numpy(float)
            lower_bounds = np.maximum(0.0, subset[lower].to_numpy(float))
            upper_bounds = np.maximum(0.0, subset[upper].to_numpy(float))
            err_low_true = np.maximum(0.0, values - lower_bounds)
            err_high_true = np.maximum(0.0, upper_bounds - values)
            epsilon = np.maximum(minimum_ci_fraction * np.maximum(values, 1.0), 1e-9)
            err_low = np.maximum(err_low_true, epsilon)
            err_high = np.maximum(err_high_true, epsilon)
            text_positions = [yi + err_hi + 0.01 * np.maximum(yi, 1.0) for yi, err_hi in zip(values, err_high)]
            ymin = float(np.nanmin(np.minimum(lower_bounds, values)))
            ymax = float(np.nanmax(np.maximum(upper_bounds, values)))
            if text_positions:
                ymax = max(ymax, float(np.nanmax(text_positions)))
            lo, hi = nice_ylim(ymin, ymax, pad_frac=0.15, floor=0.0)
            ax.errorbar(x, values, yerr=[err_low, err_high], fmt="o", markersize=4, markeredgewidth=0.8, elinewidth=1.6, capsize=5)
            ax.set_xticks(x)
            ax.set_xticklabels(subset["format"])
            ax.set_ylim(lo, hi)
            ax.yaxis.grid(True, alpha=0.35)
            if c == 0:
                ax.set_ylabel(ylabel)
            for xi, yi, text_y in zip(x, values, text_positions):
                clipped = min(text_y, hi - 0.02 * (hi - lo))
                if xi == 0 and len(x) > 1:
                    align = "left"
                elif xi == len(x) - 1 and len(x) > 1:
                    align = "right"
                else:
                    align = "center"
                ax.text(xi, clipped, f"{yi:.3g}", ha=align, va="bottom", fontsize=8, clip_on=True)
    fig.suptitle(ylabel, fontsize=12)
    fig.tight_layout(rect=(0.03, 0.03, 1, 0.96))
    plt.savefig(FIGURE_DIR / f"{prefix}.png", dpi=220)
    plt.savefig(FIGURE_DIR / f"{prefix}.pdf")
    plt.close(fig)


In [ ]:
request_intensity = compute_request_intensity_table(DATA_DIR)
if not request_intensity.empty:
    request_intensity.to_csv(TABLE_DIR / "tab_5_4_request_intensity.csv", index=False)
    plot_request_intensity_series(
        request_intensity,
        metric="GETs_per_rep",
        lower="GETs_per_rep_lo",
        upper="GETs_per_rep_hi",
        prefix="fig_5_4_reqintensity_perRep_point_ci",
        ylabel="GETs per repetition",
    )
    plot_request_intensity_series(
        request_intensity,
        metric="GETs_per_GiB",
        lower="GETs_per_GiB_lo",
        upper="GETs_per_GiB_hi",
        prefix="fig_5_4_reqintensity_perGiB_point_ci",
        ylabel="GETs per GiB",
    )
    plot_request_intensity_grid(
        request_intensity,
        metric="GETs_per_rep",
        lower="GETs_per_rep_lo",
        upper="GETs_per_rep_hi",
        ylabel="GETs per repetition",
        prefix="fig_5_4_reqintensity_perRep_grid",
        minimum_ci_fraction=0.003,
    )
    plot_request_intensity_grid(
        request_intensity,
        metric="GETs_per_GiB",
        lower="GETs_per_GiB_lo",
        upper="GETs_per_GiB_hi",
        ylabel="GETs per GiB",
        prefix="fig_5_4_reqintensity_perGiB_grid",
        minimum_ci_fraction=0.003,
    )


# CloudWatch Latencies


In [ ]:
def aggregate_cloudwatch_latencies(data_dir: Path) -> pd.DataFrame:
    records: list[dict[str, object]] = []
    for path in data_dir.iterdir():
        if not path.is_file() or path.suffix != ".json" or "_cloudwatch" not in path.name:
            continue
        meta = parse_cloudwatch_meta(path.name)
        if not meta:
            continue
        with open(path, "r") as handle:
            payload = json.load(handle)
        records.append({
            "format": meta["fmt"],
            "codec": meta["codec"],
            "pattern": meta["pattern"],
            "GetRequestsSum": float(payload.get("GetRequestsSum", np.nan)),
            "BytesDownloadedSum": float(payload.get("BytesDownloadedSum", np.nan)),
            "FirstByteLatencyAverage": float(payload.get("FirstByteLatencyAverage", np.nan)),
            "FirstByteLatencyP95": float(payload.get("FirstByteLatencyP95", np.nan)),
            "TotalRequestLatencyAverage": float(payload.get("TotalRequestLatencyAverage", np.nan)),
            "TotalRequestLatencyP95": float(payload.get("TotalRequestLatencyP95", np.nan)),
            "source": path.name,
        })
    sessions = pd.DataFrame(records)
    if sessions.empty:
        columns = [
            "format",
            "codec",
            "pattern",
            "sessions",
            "GetRequestsSum_total",
            "BytesDownloadedSum_total",
            "FirstByteLatencyAverage_ms",
            "FirstByteLatencyP95_ms",
            "TotalRequestLatencyAverage_ms",
            "TotalRequestLatencyP95_ms",
        ]
        pd.DataFrame(columns=columns).to_csv(TABLE_DIR / "tab_5_5_cw_latencies.csv", index=False)
        return sessions
    def reducer(group: pd.DataFrame) -> pd.Series:
        weights = group["GetRequestsSum"].to_numpy(float)
        weighted_sum = float(np.nansum(weights)) if np.isfinite(np.nansum(weights)) else 0.0
        def weighted_average(series: pd.Series) -> float:
            if weighted_sum > 0 and np.isfinite(series).all():
                return float(np.average(series.to_numpy(float), weights=weights))
            return float(series.mean())
        return pd.Series({
            "sessions": int(len(group)),
            "GetRequestsSum_total": float(group["GetRequestsSum"].sum()),
            "BytesDownloadedSum_total": float(group["BytesDownloadedSum"].sum()),
            "FirstByteLatencyAverage_ms": weighted_average(group["FirstByteLatencyAverage"]),
            "FirstByteLatencyP95_ms": float(group["FirstByteLatencyP95"].mean()),
            "TotalRequestLatencyAverage_ms": weighted_average(group["TotalRequestLatencyAverage"]),
            "TotalRequestLatencyP95_ms": float(group["TotalRequestLatencyP95"].mean()),
        })
    aggregated = sessions.groupby(["format", "codec", "pattern"]).apply(reducer).reset_index()
    aggregated = aggregated.sort_values(["codec", "pattern", "format"]).reset_index(drop=True)
    rounded = aggregated.copy()
    for column in [
        "FirstByteLatencyAverage_ms",
        "FirstByteLatencyP95_ms",
        "TotalRequestLatencyAverage_ms",
        "TotalRequestLatencyP95_ms",
    ]:
        rounded[column] = rounded[column].map(lambda x: np.nan if pd.isna(x) else round(float(x), 3))
    rounded.to_csv(TABLE_DIR / "tab_5_5_cw_latencies.csv", index=False)
    return aggregated


def plot_latency_grid(frame: pd.DataFrame, average_col: str, p95_col: str, title: str, prefix: str, ylabel: str) -> None:
    fig, axes = plt.subplots(nrows=len(CODECS), ncols=len(PATTERNS), figsize=(9.5, 9.0), sharex=False, sharey=False)
    width = 0.35
    for r, codec in enumerate(CODECS):
        for c, pattern in enumerate(PATTERNS):
            ax = axes[r, c] if len(CODECS) > 1 else axes[c]
            subset = frame[(frame["codec"] == codec) & (frame["pattern"] == pattern)].set_index("format").reindex(FORMATS).reset_index()
            subset = subset.dropna(subset=[average_col, p95_col])
            ax.set_title(f"{codec} / {pattern}", fontsize=10)
            if subset.empty:
                ax.axis("off")
                continue
            x = np.arange(len(subset))
            avg_vals = subset[average_col].to_numpy(float)
            p95_vals = subset[p95_col].to_numpy(float)
            ax.bar(x - width / 2, avg_vals, width, label="avg")
            ax.bar(x + width / 2, p95_vals, width, label="p95")
            ymin = float(np.nanmin(np.minimum(avg_vals, p95_vals)))
            ymax = float(np.nanmax(np.maximum(avg_vals, p95_vals)))
            lo, hi = nice_ylim(ymin, ymax, pad_frac=0.12, floor=0.0)
            ax.set_ylim(lo, hi)
            ax.set_xticks(x)
            ax.set_xticklabels(subset["format"])
            if c == 0:
                ax.set_ylabel(ylabel)
            ax.yaxis.grid(True, alpha=0.35)
            for xi, yi in zip(x - width / 2, avg_vals):
                ax.text(xi, yi, f"{yi:.1f}", ha="center", va="bottom", fontsize=8)
            for xi, yi in zip(x + width / 2, p95_vals):
                ax.text(xi, yi, f"{yi:.1f}", ha="center", va="bottom", fontsize=8)
            if r == 0 and c == len(PATTERNS) - 1:
                ax.legend(loc="upper right", fontsize=9)
    fig.suptitle(title, fontsize=12)
    fig.tight_layout(rect=(0.03, 0.03, 1, 0.96))
    plt.savefig(FIGURE_DIR / f"{prefix}.png", dpi=220)
    plt.savefig(FIGURE_DIR / f"{prefix}.pdf")
    plt.close(fig)


In [ ]:
cw_aggregated = aggregate_cloudwatch_latencies(DATA_DIR)
if not cw_aggregated.empty:
    plot_latency_grid(
        cw_aggregated,
        average_col="FirstByteLatencyAverage_ms",
        p95_col="FirstByteLatencyP95_ms",
        title="S3 FirstByteLatency: avg vs p95",
        prefix="fig_5_5_firstbyte_grid",
        ylabel="FirstByte latency (ms)",
    )
    plot_latency_grid(
        cw_aggregated,
        average_col="TotalRequestLatencyAverage_ms",
        p95_col="TotalRequestLatencyP95_ms",
        title="S3 TotalRequestLatency: avg vs p95",
        prefix="fig_5_5_total_grid",
        ylabel="TotalRequest latency (ms)",
    )
